# Baseline modele: LSTM i Transformer

Notebook definiuje pipeline do porównania baseline modeli dla rozpoznawania krótkich komend głosowych. Nie uruchamia eksperymentów automatycznie; konfiguracje, modele, dataset, trening i zapis wyników znajdują się w modułach `scripts`.

In [1]:
import pandas as pd

from scripts import (
    DataFixedParams,
    DataGridParams,
    Experiment,
    FeatureFixedParams,
    FitFixedParams,
    FitGridParams,
    LABEL_ORDER,
    ModelGridParams,
    experiment_grid_dataframe,
    prepare_experiment_datasets,
    train_experiment,
)

LABEL_ORDER

('yes',
 'no',
 'up',
 'down',
 'left',
 'right',
 'on',
 'off',
 'stop',
 'go',
 'unknown',
 'silence')

## Konfiguracja eksperymentu


In [2]:
# Wszystkie parametry eksperymentu są zebrane w tej komórce.
# Konfiguracja poniżej jest małym, ale użytecznym testem poprawności treningu.
data_fixed = DataFixedParams()
feature_fixed = FeatureFixedParams(
    n_mels=64,
    n_fft=512,
    hop_length=160,
    normalize=True,
)

data_grid = DataGridParams(
    train_fraction=0.1,
    validation_fraction=0.1,
    test_fraction=0.1,
    unknown_fraction=0.01,
    silence_examples_per_split=50,
    sampling_strategy="natural",
    seed=42,
)

model_grid = ModelGridParams(
    # model_type=["lstm", "transformer"],
    model_type=["transformer"],
    dropout=0.15,
    lstm_hidden_size=64,
    lstm_layers=2,
    lstm_bidirectional=True,
    transformer_d_model=64,
    transformer_heads=4,
    transformer_layers=2,
    transformer_ff_dim=128,
)

fit_fixed = FitFixedParams(
    device="auto",
    use_tqdm=True,
    progress_backend="terminal",
    verbose=True,
    early_stopping=True,
    early_stopping_patience=3,
    early_stopping_min_delta=0.001,
)
fit_grid = FitGridParams(
    epochs=5,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

baseline_experiment = Experiment(
    name="small_functional_baseline_lstm_transformer",
    data_fixed=data_fixed,
    data_grid=data_grid,
    feature_fixed=feature_fixed,
    model_grid=model_grid,
    fit_fixed=fit_fixed,
    fit_grid=fit_grid,
)

experiment_grid_dataframe(baseline_experiment)


,experiment,data.train_fraction,data.validation_fraction,data.test_fraction,data.unknown_fraction,data.silence_examples_per_split,data.sampling_strategy,data.seed,model.model_type,model.dropout,...,model.lstm_layers,model.lstm_bidirectional,model.transformer_d_model,model.transformer_heads,model.transformer_layers,model.transformer_ff_dim,fit.epochs,fit.batch_size,fit.learning_rate,fit.weight_decay
0,small_functional_baseline_lstm_transformer,0.1,0.1,0.1,0.01,50,natural,42,lstm,0.15,...,2,True,64,4,2,128,5,32,0.001,0.0001
1,small_functional_baseline_lstm_transformer,0.1,0.1,0.1,0.01,50,natural,42,transformer,0.15,...,2,True,64,4,2,128,5,32,0.001,0.0001


## Uruchomienie eksperymentu

Najpierw przygotuj dane i sprawdź obiekt `prepared_data`. Dopiero kolejna komórka uruchamia trening oraz końcowy test.


In [5]:
prepared_data = prepare_experiment_datasets(baseline_experiment)


Building dataset
  -> samples | train=2190 | validation=309 | test=310
     class        train  validation  test
     down          185          27    26
     go            187          26    26
     left          184          25    27
     no            186          27    26
     off           184          26    27
     on            187          26    25
     right         186          26    26
     silence         5           5     5
     stop          189          25    25
     unknown       326          43    43
     up            185          26    28
     yes           186          27    26


In [4]:
results = train_experiment(baseline_experiment, prepared_data)
results


Starting experiment: small_functional_baseline_lstm_transformer

Configuration run 1/2:
DATA (variable):
  - train_fraction: 0.1
  - validation_fraction: 0.1
  - test_fraction: 0.1
  - unknown_fraction: 0.01
  - silence_examples_per_split: 50
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: lstm
  - dropout: 0.15
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 5
  - batch_size: 32
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cpu


Training: 100%|██████████| 5/5 [01:28<00:00, 17.79s/it, loss=0.9840, lr=0.001, val_acc=0.6861, val_loss=0.9894]


Training finished in 90.66 seconds



Configuration run 2/2:
DATA (variable):
  - train_fraction: 0.1
  - validation_fraction: 0.1
  - test_fraction: 0.1
  - unknown_fraction: 0.01
  - silence_examples_per_split: 50
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: transformer
  - dropout: 0.15
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 5
  - batch_size: 32
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cpu


Training: 100%|██████████| 5/5 [01:46<00:00, 21.38s/it, loss=1.0194, lr=0.001, val_acc=0.6958, val_loss=0.9669]


Training finished in 108.50 seconds



Experiment finished | total runs = 2


,run,best_epoch,epochs_trained,stopped_early,train_loss,train_accuracy,validation_loss,validation_accuracy,test_loss,test_accuracy
0,lstm_train0_1_val0_1_test0_1_lr0_001_seed42,5,5,False,0.983989,0.687215,0.989350,0.686084,1.192971,0.577419
1,trfm_train0_1_val0_1_test0_1_lr0_001_seed42,5,5,False,1.019355,0.650685,0.966877,0.695793,1.035343,0.641935
